In [ ]:
# ============================================
# train_gnn.py — Time-Varying Alpha Generator
# ============================================

import os
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GATv2Conv
from sklearn.preprocessing import StandardScaler

# --------------------------
# 🔧 Paths
# --------------------------
save_path = "data/processed"
factor_path = os.path.join(save_path, "factor_library.parquet")
adj_path = os.path.join(save_path, "graph_adj.npy")

# --------------------------
# 📂 Load Data
# --------------------------
factors = pd.read_parquet(factor_path)
adj = np.load(adj_path)

print(f"✅ Loaded factor data: {factors.shape}")
print(f"✅ Loaded adjacency matrix: {adj.shape}")

# --------------------------
# 🧱 Prepare Graph Structure
# --------------------------
edge_index = torch.tensor(np.array(np.nonzero(adj > 0)), dtype=torch.long)
tickers = factors['ticker'].unique()
ticker_to_idx = {t: i for i, t in enumerate(tickers)}

# --------------------------
# ⚙️ Define GNN Model
# --------------------------
class GNN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GATv2Conv(in_channels, hidden_channels, heads=4, dropout=0.2)
        self.conv2 = GATv2Conv(hidden_channels * 4, out_channels, heads=1, concat=False, dropout=0.2)
    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.elu(x)
        x = self.conv2(x, edge_index)
        return x

# --------------------------
# 🚀 Training Setup
# --------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
in_channels = 1  # one feature per ticker (e.g., return)
hidden_channels = 32
out_channels = 16
lr = 0.005
epochs = 100

model = GNN(in_channels, hidden_channels, out_channels).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

# --------------------------
# 🧠 Train over Time
# --------------------------
unique_dates = sorted(factors['date'].unique())
all_alpha_rows = []

for d in unique_dates:
    # Use returns up to this date as node features
    snapshot = factors[factors['date'] <= d]
    X = snapshot.groupby('ticker')['ret'].last().reindex(tickers).fillna(0).values.reshape(-1, 1)

    scaler = StandardScaler()
    X = scaler.fit_transform(X)
    X = torch.tensor(X, dtype=torch.float).to(device)

    data = Data(x=X, edge_index=edge_index.to(device))

    # Train autoencoder on this snapshot
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        _ = model(data.x, data.edge_index)  # forward pass only
        optimizer.step()

    model.eval()
    with torch.no_grad():
        # embeddings = model(data.x, data.edge_index).cpu().numpy()
        embeddings = model(data.x, data.edge_index).cpu().numpy()

    # Take the first latent dimension as the alpha
    alpha = embeddings[:, 0]
    date_rows = pd.DataFrame({
        'date': d,
        'ticker': tickers,
        'alpha': alpha
    })
    all_alpha_rows.append(date_rows)

# --------------------------
# 💾 Save Outputs
# --------------------------
alpha_predictions = pd.concat(all_alpha_rows, ignore_index=True)
alpha_predictions.to_parquet(os.path.join(save_path, "alpha_predictions.parquet"))
print("✅ Saved time-varying alpha predictions to alpha_predictions.parquet")

print("🎯 Training complete — ready for backtest.")
